DATA QUALITY VERIFICATION
_____________________________

BUSINESS QUESTION

1. Which companies laid off staff more than once?
2. For those companies, how much time passed between their layoff rounds?
3. Were the rounds close together (an ongoing crisis) or spread far apart (separate, unrelated downturns)?


In [1]:
import pandas as pd

Loading cleaned dataset

In [3]:
df = pd.read_csv("D:\DATA ANALYSIS\MySQL data project(A)\Data\Raw\layoff2.csv", parse_dates=["date"])
df_with_dates = df[df["date"].notna()].copy()

Count number of layoff events each company had

In [4]:
event_counts = df_with_dates.groupby("company").size()
repeat_companies = event_counts[event_counts > 1].index
print(f"Number of companies with more than one layoff event: {len(repeat_companies)}")

Number of companies with more than one layoff event: 282


Building a dataframe containing only repeat-layoff companies

In [5]:
df_repeats = df_with_dates[df_with_dates["company"].isin(repeat_companies)].copy()
df_repeats = df_repeats.sort_values(by=["company", "date"])

Calculating days between each company's layoff events

In [7]:
df_repeats["days_since_previous_layoff"] = (
    df_repeats.groupby("company")["date"].diff().dt.days
)


Building a per-company summary table

In [10]:
company_summary = df_repeats.groupby("company").agg(
    number_of_layoff_events=("date", "count"),
    first_layoff_date=("date", "min"),
    last_layoff_date=("date", "max"),
    total_laid_off_all_events=("total_laid_off", "sum"),
    average_days_between_layoffs=("days_since_previous_layoff", "mean"),
)

company_summary = company_summary.reset_index()
company_summary["average_days_between_layoffs"] = company_summary[
    "average_days_between_layoffs"
].round(1)

company_summary = company_summary.sort_values(
    by="number_of_layoff_events", ascending=False
)

print("\nTop repeat-layoff companies:")
print(company_summary.head(10).to_string(index=False))


Top repeat-layoff companies:
   company  number_of_layoff_events first_layoff_date last_layoff_date  total_laid_off_all_events  average_days_between_layoffs
      Loft                        6        2020-03-27       2023-03-03                       1289                         214.2
    Swiggy                        5        2020-04-21       2023-01-20                       2880                         251.0
      Uber                        5        2020-05-06       2022-09-07                       7585                         213.5
    WeWork                        5        2020-03-28       2023-01-19                       1150                         256.8
   Netflix                        4        2022-04-28       2022-09-14                        505                          46.3
       Ola                        4        2020-05-20       2023-01-13                       2800                         322.7
Salesforce                        4        2020-08-26       2023-01-04    

Exporting both tables for Power BI

In [9]:
company_summary.to_csv("repeat_layoff_companies_summary.csv", index=False)
df_repeats.to_csv("repeat_layoff_events_detail.csv", index=False)

print("\nExported: repeat_layoff_companies_summary.csv")
print("Exported: repeat_layoff_events_detail.csv")


Exported: repeat_layoff_companies_summary.csv
Exported: repeat_layoff_events_detail.csv
